# SVO Grammar Coach (Step 2) - Grammar Engine

We load the lexicon and build a small grammar engine for simple English SVO sentences.
We keep the rules explicit and educational rather than fully complete.

Plan:
- load the resources file
- define determiners
- build noun phrases (subject/object)
- conjugate regular verbs
- assemble sentences and check user input


In [1]:
import json
from pathlib import Path

# Load resources generated in Step 1
paths = [
    Path("notebooks_exc_2/svo_resources.json"),
    Path("svo_resources.json"),
]

resources = None
for p in paths:
    if p.exists():
        resources = json.loads(p.read_text())
        break

if resources is None:
    raise FileNotFoundError("svo_resources.json not found. Run Step 1 first.")

NOUNS = resources["nouns"]
ADJECTIVES = resources["adjectives"]
VERBS = resources["verbs"]

print("Loaded:", len(NOUNS), "nouns,", len(ADJECTIVES), "adjectives,", len(VERBS), "verbs")


Loaded: 75 nouns, 100 adjectives, 100 verbs


## 1. Determiners and noun phrases


In [2]:
VOWELS = set("aeiou")

POSSESSIVE = {
    "my": "my",
    "your": "your",
    "his": "his",
    "her": "her",
    "its": "its",
    "our": "our",
    "their": "their",
}


def plural_form(noun):
    return NOUNS[noun]["plural"]


def choose_indefinite(next_word):
    if not next_word:
        return "a"
    return "an" if next_word[0].lower() in VOWELS else "a"


def determiner(d_type, number, next_word=None, possessor_key=None):
    if d_type == "none":
        return ""
    if d_type == "definite":
        return "the"
    if d_type == "indefinite":
        if number == "pl":
            return ""
        return choose_indefinite(next_word)
    if d_type == "possessive":
        return POSSESSIVE.get(possessor_key or "my", "my")
    if d_type == "demonstrative":
        return "this" if number == "sg" else "these"
    raise ValueError("Unknown determiner type")


def make_noun_phrase(
    noun,
    number="sg",
    d_type="definite",
    adjective=None,
    possessor_key=None,
):
    head = noun if number == "sg" else plural_form(noun)
    next_word = adjective or head
    det = determiner(d_type, number, next_word=next_word, possessor_key=possessor_key)

    parts = []
    if det:
        parts.append(det)
    if adjective:
        parts.append(adjective)
    parts.append(head)
    return " ".join(parts)


## 2. Regular verb conjugation


In [3]:
VOWELS = set("aeiou")


def present_3sg(verb):
    if verb.endswith("y") and len(verb) >= 2 and verb[-2] not in VOWELS:
        return verb[:-1] + "ies"
    if verb.endswith(("s", "sh", "ch", "x", "z", "o")):
        return verb + "es"
    return verb + "s"


def past_regular(verb):
    if verb.endswith("e"):
        return verb + "d"
    if verb.endswith("y") and len(verb) >= 2 and verb[-2] not in VOWELS:
        return verb[:-1] + "ied"
    return verb + "ed"


def conjugate_present(verb, person, number):
    if person == 3 and number == "sg":
        return present_3sg(verb)
    return verb


def do_aux(tense, person, number):
    if tense == "past":
        return "did"
    if person == 3 and number == "sg":
        return "does"
    return "do"


def imperative_form(verb):
    return verb


def would_form():
    return "would"


## 3. Sentence builder and grammar check


In [6]:
import difflib
PRONOUNS = {
    (1, "sg"): "I",
    (2, "sg"): "you",
    (3, "sg", "m"): "he",
    (3, "sg", "f"): "she",
    (3, "sg", "n"): "it",
    (1, "pl"): "we",
    (2, "pl"): "you",
    (3, "pl", "pl"): "they",
}
def sentence_case(s):
    if not s:
        return s
    return s[0].upper() + s[1:]
def build_sentence(
    subject_phrase,
    verb_inf,
    object_phrase,
    sentence_type="affirmative",
    tense="present",
    person=3,
    number="sg",
):
    if sentence_type == "subjunctive":
        core = f"{subject_phrase} {would_form()} {verb_inf} {object_phrase}".strip()
        return sentence_case(core) + "."
    if sentence_type == "imperative":
        core = f"{imperative_form(verb_inf)} {object_phrase}".strip()
        return sentence_case(core) + "!"
    if sentence_type == "question":
        aux = do_aux(tense, person, number)
        core = f"{aux} {subject_phrase} {verb_inf} {object_phrase}".strip()
        return sentence_case(core) + "?"
    if sentence_type == "negative":
        aux = do_aux(tense, person, number)
        core = f"{subject_phrase} {aux} not {verb_inf} {object_phrase}".strip()
        return sentence_case(core) + "."
    # affirmative
    if tense == "past":
        verb_form = past_regular(verb_inf)
    else:
        verb_form = conjugate_present(verb_inf, person, number)
    core = f"{subject_phrase} {verb_form} {object_phrase}".strip()
    return sentence_case(core) + "."
def check_sentence(user_sentence, expected_sentence):
    user = user_sentence.strip()
    expected = expected_sentence.strip()
    if user == expected:
        return "Correct."
    diff = " ".join(difflib.ndiff(expected.split(), user.split()))
    return (
        "Not correct.\n"
        + f"Expected: {expected}\n"
        + f"You wrote: {user}\n"
        + f"Diff: {diff}"
    )

In [7]:
# Demo
subject = make_noun_phrase("man", number="sg", d_type="definite", adjective="big")
obj = make_noun_phrase("book", number="sg", d_type="indefinite", adjective="interesting")

print(build_sentence(subject, "order", obj, sentence_type="affirmative", tense="present", person=3, number="sg"))
print(build_sentence(subject, "order", obj, sentence_type="negative", tense="present", person=3, number="sg"))
print(build_sentence(subject, "order", obj, sentence_type="question", tense="present", person=3, number="sg"))
print(build_sentence(subject, "order", obj, sentence_type="imperative", tense="present", person=2, number="pl"))
print(build_sentence(subject, "order", obj, sentence_type="subjunctive", tense="present", person=3, number="sg"))

expected = build_sentence(subject, "order", obj, sentence_type="affirmative", tense="present", person=3, number="sg")
print(check_sentence("The big man orders an interesting book.", expected))


The big man orders an interesting book.
The big man does not order an interesting book.
Does the big man order an interesting book?
Order an interesting book!
The big man would order an interesting book.
Correct.
